In [1]:
import pandas as pd
import numpy as np

# 1. Load the new testing dataset
# Replace the path with your actual filename
df_test_new = pd.read_csv('anon-Booter_Combined.csv')

# 2. Basic Inspection
print("--- Dataset Overview ---")
print(f"Total Rows: {df_test_new.shape[0]}")
print(f"Total Columns: {df_test_new.shape[1]}")
print("\nFirst 5 Rows:")
display(df_test_new.head())

# 3. Check for specific problematic values
print("\n--- Value Integrity Check ---")
print(f"Any Null Values? {df_test_new.isnull().values.any()}")
# Checking for infinity which is common in 'Flow Packets/s' or 'Flow Bytes/s'
inf_count = np.isinf(df_test_new.select_dtypes(include=np.number)).values.sum()
print(f"Any Infinity Values? {inf_count > 0} (Count: {inf_count})")

--- Dataset Overview ---
Total Rows: 1771055
Total Columns: 84

First 5 Rows:


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,227.213.154.241-99.99.217.121-0-0-0,99.99.217.121,0,227.213.154.241,0,0,15/08/2013 05:02:41 AM,113526102,15568,1,...,0,104061946.0,0.000000e+00,104061946.0,104061946.0,6907213.0,0.000000e+00,6907213.0,6907213.0,1
1,227.213.154.241-253.80.175.55-0-0-0,253.80.175.55,0,227.213.154.241,0,0,15/08/2013 05:02:41 AM,113465880,13329,1,...,0,104021225.0,0.000000e+00,104021225.0,104021225.0,6875233.0,0.000000e+00,6875233.0,6875233.0,1
2,227.213.154.241-99.99.185.83-0-0-0,99.99.185.83,0,227.213.154.241,0,0,15/08/2013 05:02:41 AM,113597071,15301,1,...,0,49549442.0,6.653015e+07,96593362.0,2505522.0,5968615.0,1.317778e+06,6900425.0,5036805.0,1
3,227.213.154.241-101.229.203.241-0-0-0,101.229.203.241,0,227.213.154.241,0,0,15/08/2013 05:02:41 AM,113667698,15248,1,...,0,104203207.0,0.000000e+00,104203207.0,104203207.0,6903488.0,0.000000e+00,6903488.0,6903488.0,1
4,227.213.154.241-99.99.159.15-0-0-0,99.99.159.15,0,227.213.154.241,0,0,15/08/2013 05:02:41 AM,113629212,15137,1,...,0,49544501.5,6.653300e+07,96590435.0,2498568.0,5992330.0,1.295855e+06,6908638.0,5076022.0,1



--- Value Integrity Check ---
Any Null Values? True
Any Infinity Values? False (Count: 0)


In [2]:
df_test_new['Label'].value_counts()

Label
1    1771055
Name: count, dtype: int64

In [3]:
import joblib
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load the Brain
imputer_bundle = joblib.load("regression_imputer.pkl")
scaler = joblib.load("scaler.pkl")


# Load the Random Forest instead of the CNN
rf_model_full = joblib.load("Full_RF_Classifier.pkl") # Or "Full_RF_Classifier.pkl" if you dropped the AE

# 2. Verify the snake_case names
print("--- Features expected by the model ---")
expected_features = list(scaler.feature_names_in_)
print(expected_features)

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


--- Features expected by the model ---
['src_port', 'dst_port', 'protocol', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'flow_byts_s', 'flow_pkts_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_tot', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_tot', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_len', 'bwd_header_len', 'fwd_pkts_s', 'bwd_pkts_s', 'pkt_len_min', 'pkt_len_max', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'cwe_flag_count', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'fwd_byts_b_avg', 'fwd_pk

In [4]:
# Check the raw column names
print("Raw columns in the new dataset:")
print(df_test_new.columns.tolist())

Raw columns in the new dataset:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg

In [5]:
df_test_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1771055 entries, 0 to 1771054
Data columns (total 84 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Flow ID            object 
 1   Src IP             object 
 2   Src Port           int64  
 3   Dst IP             object 
 4   Dst Port           int64  
 5   Protocol           int64  
 6   Timestamp          object 
 7   Flow Duration      int64  
 8   Tot Fwd Pkts       int64  
 9   Tot Bwd Pkts       int64  
 10  TotLen Fwd Pkts    float64
 11  TotLen Bwd Pkts    float64
 12  Fwd Pkt Len Max    float64
 13  Fwd Pkt Len Min    float64
 14  Fwd Pkt Len Mean   float64
 15  Fwd Pkt Len Std    float64
 16  Bwd Pkt Len Max    float64
 17  Bwd Pkt Len Min    float64
 18  Bwd Pkt Len Mean   float64
 19  Bwd Pkt Len Std    float64
 20  Flow Byts/s        float64
 21  Flow Pkts/s        float64
 22  Flow IAT Mean      float64
 23  Flow IAT Std       float64
 24  Flow IAT Max       float64
 25  Flow IAT Min      

In [6]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df_test_new.columns = df_test_new.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores
df_test_new.columns = df_test_new.columns.str.replace(' ', '_', regex=False)
df_test_new.columns = df_test_new.columns.str.replace('/', '_', regex=False)
df_test_new.columns = df_test_new.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df_test_new.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df_test_new.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [7]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df_test_new.drop(columns=[col for col in identifiers if col in df_test_new.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df_test_new.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [8]:
# Rename the specific columns to match the training feature set
df_test_new.rename(columns={
    'bwd_iat_total': 'bwd_iat_tot',
    'fwd_packets_s': 'fwd_pkts_s',
    'bwd_packets_s': 'bwd_pkts_s'
}, inplace=True)

# Re-extract X_test_new to ensure it picks up the renamed columns
X_test_new = df_test_new.drop(columns=['label'])

print("Renaming successful.")
print(f"Is 'bwd_iat_tot' now in X? {'bwd_iat_tot' in X_test_new.columns}")
print(f"Is 'fwd_pkts_s' now in X? {'fwd_pkts_s' in X_test_new.columns}")
print(f"Is 'bwd_pkts_s' now in X? {'bwd_pkts_s' in X_test_new.columns}")
print(f"Is 'bwd_iat_total' now in X? {'bwd_iat_total' in X_test_new.columns}")
print(f"Is 'fwd_packets_s' now in X? {'fwd_packets_s' in X_test_new.columns}")
print(f"Is 'bwd_packets_s' now in X? {'bwd_packets_s' in X_test_new.columns}")

Renaming successful.
Is 'bwd_iat_tot' now in X? True
Is 'fwd_pkts_s' now in X? True
Is 'bwd_pkts_s' now in X? True
Is 'bwd_iat_total' now in X? False
Is 'fwd_packets_s' now in X? False
Is 'bwd_packets_s' now in X? False


In [9]:
imputer_models = imputer_bundle['models']
COLS_WONA = imputer_bundle['cols_wona']
COLS_WNA = imputer_bundle['cols_wna']

# Replace Inf with NaN (same as training)
df_test_new.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Remaining NaNs:", df_test_new.isna().sum().sum())

for target_col, model in imputer_models.items():
    if target_col not in df_test_new.columns:
        continue  # safety check

    predict_data = df_test_new[df_test_new[target_col].isna()]

    if not predict_data.empty:
        # Ensure predictor columns exist
        X = predict_data[COLS_WONA]

        predictions = model.predict(X)
        df_test_new.loc[df_test_new[target_col].isna(), target_col] = predictions

print("Remaining NaNs:", df_test_new.isna().sum().sum())

Remaining NaNs: 1340
Remaining NaNs: 0


In [10]:
# 1. Identify rows with Infinity
inf_mask = np.isinf(df_test_new.select_dtypes(include=np.number)).any(axis=1)
inf_count = inf_mask.sum()

# 2. Identify rows with Nulls
null_mask = df_test_new.isnull().any(axis=1)
null_count = null_mask.sum()

print(f"Rows with Infinity: {inf_count}")
print(f"Rows with Nulls: {null_count}")
print(f"Final testing row count: {df_test_new.shape[0]}")

Rows with Infinity: 0
Rows with Nulls: 0
Final testing row count: 1771055


In [11]:
# 1. Extract the labels into y_test_new
y_test_new = df_test_new['label'].copy()

# 2. Extract the features into X_test_new by dropping the label column
X_test_new = df_test_new.drop(columns=['label'])

print("Separation Complete.")
print(f"Features Shape (X): {X_test_new.shape}")
print(f"Labels Shape (y): {y_test_new.shape}")

Separation Complete.
Features Shape (X): (1771055, 79)
Labels Shape (y): (1771055,)


In [12]:
# 1. Get the list of features the scaler expects
expected = list(scaler.feature_names_in_)

# 2. Get the list of features currently in your X_test_new
current = list(X_test_new.columns)

# 3. Find the differences
missing_in_current = [f for f in expected if f not in current]
extra_in_current = [f for f in current if f not in expected]

print("--- Column Name Check ---")
print(f"Features missing from your Data: {missing_in_current}")
print(f"Features in your Data that shouldn't be there: {extra_in_current}")

--- Column Name Check ---
Features missing from your Data: []
Features in your Data that shouldn't be there: []


In [13]:
print("Unique labels found in new data:")
print(y_test_new.unique())

Unique labels found in new data:
[1]


In [14]:
# 1. Transform the data using the loaded Min-Max Scaler
# This will output float64 by default if the scaler was fitted with float64
X_test_scaled = scaler.transform(X_test_new)

# 2. Ensure it is explicitly float64 (matching your training)
X_test_scaled = X_test_scaled.astype('float64')

print("Scaling Complete.")
print(f"Data Type: {X_test_scaled.dtype}")
print(f"Scaled Shape: {X_test_scaled.shape}")

Scaling Complete.
Data Type: float64
Scaled Shape: (1771055, 79)


In [15]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Starting Full Random Forest Predictions...")

# 1. Generate direct class predictions from the RF model
# Using the 79 scaled features directly
y_pred = rf_model_full.predict(X_test_scaled) 

# 2. Use the original y_test_new for comparison
print(f"Final Test Accuracy on New Dataset: {accuracy_score(y_test_new, y_pred):.5f}")

print("\nClassification Report:")
# We explicitly set labels=[0, 1] to handle datasets that might only contain attacks
print(classification_report(y_test_new, y_pred, target_names=['BENIGN', 'ATTACK'], labels=[0, 1], zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_new, y_pred, labels=[0, 1]))

Starting Full Random Forest Predictions...


[Parallel(n_jobs=48)]: Using backend ThreadingBackend with 48 concurrent workers.
[Parallel(n_jobs=48)]: Done 100 out of 100 | elapsed:    1.1s finished


Final Test Accuracy on New Dataset: 0.95516

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.00      0.00      0.00         0
      ATTACK       1.00      0.96      0.98   1771055

    accuracy                           0.96   1771055
   macro avg       0.50      0.48      0.49   1771055
weighted avg       1.00      0.96      0.98   1771055


Confusion Matrix:
[[      0       0]
 [  79418 1691637]]
